In [ ]:
# PyTorch functions/methods helpers

# 8.6.1
class ResidualBlock(nn.Module):

    def __init__(self, in_channels, out_channels, stride=1, use_projection=False):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels)
        )
        if use_projection:
            self.shortcut = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False)
        else:
            self.shortcut = nn.Identity() # Returns the input unchanged: shortcut(X) = X

    def forward(self, X):
        return F.relu(self.main(X) + self.shortcut(X))

# 8.6.4
tiny_resnet = nn.Sequential(
    nn.Conv2d(1, 8, kernel_size=3, padding=1), nn.BatchNorm2d(8), nn.ReLU(),
    ResidualBlock(8, 8),                                                  # I already have (2, 8, 32, 32). Let's give the network 2 convolutions to refine those features, but don't force it to completely replace them
    ResidualBlock(8, 16, stride=2, use_projection=True),                  # Use projection if in_channels != out_channels OR stride != 1 (in here, we reduced the spatial resolution and increased the # of channels)
    ResidualBlock(16, 16),                                                # Now that we're (2, 16, 16, 16), let's refine those features further while keeping an easy path for the existing features
    nn.AdaptiveAvgPool2d((1, 1)),
    nn.Flatten(),
    nn.Linear(16, 10),
)

# 8.6.5
grouped_conv = nn.Conv2d(16, 32, kernel_size=3, padding=1, groups=4, bias=False) # Split channels into 4 groups (or 16/4=4 input_channels and 32/4=8 output_channels per group)


ResNet changes the default function a block has to learn.
* Instead of asking stacked layers to learn a full transformation from scratch, a **residual block learns an update added to the input**.
* ResNeXt extends this family with grouped convolutions for a different capacity-compute tradeoff.

# How to use this notebook

* Run the notebook from top to bottom in a clean kernel.

* The code uses small synthetic tensors so that architecture mechanics can be inspected without downloads, `torchvision`, ImageNet-scale images, or long training runs.

* Before important cells, predict the shape, parameter count, or failure mode, then read the assertions as executable contracts.

# You are done when you can

- explain why residual blocks make identity mappings easier
- implement residual addition with and without projection
- trace shape changes through a tiny ResNet
- explain grouped convolution in ResNeXt terms
- debug residual addition when the two paths have incompatible shapes

In [2]:
import math

import torch
from torch import nn
from torch.nn import functional as F

torch.manual_seed(0)
torch.set_printoptions(precision=4, sci_mode=False)

def shape(x):
    return tuple(x.shape)

def count_parameters(module):
    return sum(p.numel() for p in module.parameters())

def trace_module_shapes(module, X):
    rows = []
    current = X
    for name, layer in module.named_children():
        current = layer(current)
        rows.append((name, layer.__class__.__name__, shape(current)))
    return rows, current

# 8.6.0 The Problem This Notebook Solves

Deeper networks should be more expressive, but they can become harder to optimize.

ResNet's core idea is to make a block learn a residual update:

```text
output = input + learned_update(input)
```

* If the best behavior is close to "do nothing", the learned update can move toward zero.
* This gives the architecture an easier path to identity mappings than asking several layers to directly learn identity from scratch.

# 8.6.1 Residual Addition Requires Matching Shapes

Addition is stricter than concatenation.

To compute `Y + X`, both tensors must have the same shape or be broadcast-compatible.

In ResNet blocks, the intended case is same shape:

```text
main path output shape == shortcut path output shape
```

The block below keeps channel count and spatial size unchanged.

In [3]:
class ResidualBlock(nn.Module):

    def __init__(self, in_channels, out_channels, stride=1, use_projection=False):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels)
        )
        if use_projection:
            self.shortcut = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False)
        else:
            self.shortcut = nn.Identity() # Returns the input unchanged: shortcut(X) = X

    def forward(self, X):
        return F.relu(self.main(X) + self.shortcut(X))

block = ResidualBlock(8, 8) # in_channels = 8, out_channels = 8
X = torch.randn(2, 8, 16, 16)

# Layer 0 = output size of 16 + 2*1 - 3 + 1 = 16 for shape of (batch, out_channels, height, width) or (2, 8, 16, 16)
# Layer 1 BatchNorm2d normalizes each channel over batch, height, width; updates running_mean/running_var during training; has learnable gamma/beta
# Layer 2 ReLU
# Layer 3 = output size of 16 + 2*1 - 3 + 1 = 16 for shape of (batch, out_channels, height, width) or (2, 8, 16, 16)
# Layer 4 BatchNorm2d normalizes each channel over batch, height, width; updates running_mean/running_var during training; has learnable gamma/beta
# self.shortcut = nn.Identity() = X

Y = block(X)

print("output:", shape(Y))
assert shape(Y) == shape(X)

output: (2, 8, 16, 16)


# 8.6.2 Projection Shortcuts Change Shape Deliberately

When a residual stage changes channel count or spatial size, the shortcut path must change shape too.

A 1 by 1 convolution can project the input into the required shape.

Here the main path uses stride 2 and changes channels from 8 to 16. The shortcut must do the same.

In [4]:
projection_block = ResidualBlock(8, 16, stride=2, use_projection=True) # in_channels = 8, out_channels = 16
X = torch.randn(2, 8, 16, 16)
Y = projection_block(X)

shortcut_Y = projection_block.shortcut(X) # (2, 16, 8, 8)
main_Y = projection_block.main(X) # (2, 16, 8, 8)

# Layer 0 = output size of floor((16 + 2*1 - 3)/2 + 1) = 8 for shape of (batch, out_channels, height, width) or (2, 16, 8, 8)
# Layer 1 BatchNorm2d normalizes each channel over batch, height, width; updates running_mean/running_var during training; has learnable gamma/beta
# Layer 2 ReLU
# Layer 3 = output size of 8 + 2*1 - 3 + 1 = 8 for shape of (batch, out_channels, height, width) or (2, 16, 8, 8)
# Layer 4 BatchNorm2d normalizes each channel over batch, height, width; updates running_mean/running_var during training; has learnable gamma/beta
# self.shortcut = output size of floor((16 - 1)/2 + 1) = 8 for shape of (batch, out_channels, height, width) or (2, 16, 8, 8)

print("main path:", shape(main_Y))
print("shortcut:", shape(shortcut_Y))
print("block output:", shape(Y))

assert shape(main_Y) == shape(shortcut_Y)
assert shape(Y) == (2, 16, 8, 8)

main path: (2, 16, 8, 8)
shortcut: (2, 16, 8, 8)
block output: (2, 16, 8, 8)


# 8.6.3 Identity Is Easy When the Residual Update Is Zero

This tiny block strips away convolution and batch normalization to isolate the function-class idea.

If the learned update is zero, the residual block returns the input, up to the final activation.

For nonnegative inputs, `relu(X + 0)` equals `X`.

In [5]:
class ZeroUpdateResidual(nn.Module):

    def forward(self, X):
        update = torch.zeros_like(X) # Fill in existing shape of X with zeros (unlike torch.zeros() which defines the shape as arguments)
        return F.relu(X + update)

X = torch.rand(2, 3, 4, 4)
Y = ZeroUpdateResidual()(X)

print("max difference:", (Y - X).abs().max().item()) # update = 0, so X + update = X; because X >= 0, ReLU(X) = X
assert torch.equal(Y, X)


max difference: 0.0


# 8.6.4 A Tiny ResNet Uses Residual Blocks as Stages

A ResNet is not just one residual block. It uses stages:

```text
stem -> residual stage -> residual stage -> global average pool -> classifier
```

The first block of a new stage often downsamples and changes channel count with a projection shortcut.

Later blocks in the same stage keep shape.

In [6]:
tiny_resnet = nn.Sequential(
    nn.Conv2d(1, 8, kernel_size=3, padding=1), nn.BatchNorm2d(8), nn.ReLU(),
    ResidualBlock(8, 8),                                                  # I already have (2, 8, 32, 32). Let's give the network 2 convolutions to refine those features, but don't force it to completely replace them
    ResidualBlock(8, 16, stride=2, use_projection=True),                  # Use projection if in_channels != out_channels OR stride != 1 (in here, we reduced the spatial resolution and increased the # of channels)
    ResidualBlock(16, 16),                                                # Now that we're (2, 16, 16, 16), let's refine those features further while keeping an easy path for the existing features
    nn.AdaptiveAvgPool2d((1, 1)),
    nn.Flatten(),
    nn.Linear(16, 10),
)

X = torch.randn(2, 1, 32, 32)

# Layer 0 = output size of ((32 + 2*1 - 3) + 1) = 32 for shape of (batch, out_channels, height, width) or (2, 8, 32, 32)
# Layer 1 = BatchNorm2d normalizes each channel over batch, height, width; updates running_mean/running_var during training; has learnable gamma/beta
# Layer 2 = ReLU
# Layer 3 = shape of (2, 8, 32, 32) unchanged, but performed 2 BatchNorm2d and 1 ReLU in the process
# Layer 4 = output size of ((32 + 2* 1 - 3)/2 + 1) = 16 for shape of (batch, out_channels, height, width) or (2, 16, 16, 16), alongside 2 BatchNorm2d and 1 ReLU
# Layer 5 = (2, 16, 16, 16)
# Layer 6 = (2, 16, 1, 1) as height and width are now averaged per dimension (across columns)
# Layer 7 = (2, 16)
# Layer 8 = Layer 7 output of (2, 16) @ linear.weight.T shape of (16, 10) = (2, 10)

rows, logits = trace_module_shapes(tiny_resnet, X)
for row in rows:
    print(row)

assert shape(logits) == (2, 10)

('0', 'Conv2d', (2, 8, 32, 32))
('1', 'BatchNorm2d', (2, 8, 32, 32))
('2', 'ReLU', (2, 8, 32, 32))
('3', 'ResidualBlock', (2, 8, 32, 32))
('4', 'ResidualBlock', (2, 16, 16, 16))
('5', 'ResidualBlock', (2, 16, 16, 16))
('6', 'AdaptiveAvgPool2d', (2, 16, 1, 1))
('7', 'Flatten', (2, 16))
('8', 'Linear', (2, 10))


## What are the convolutional layers in a residual block actually learning?

Suppose you already have some useful features:

$$
X = \text{useful features I've already learned}
$$

Now you want the next part of the network to **improve** those features.

Maybe the best transformation is:

$$
X \rightarrow \text{almost the same features, but with some additional useful information}
$$

---

### Ordinary CNN

An ordinary CNN has to learn the entire desired transformation:

$$
Y = H(X)
$$

In other words, the convolutional layers have to figure out the complete output representation.

---

### Residual block

A residual block instead learns a **residual**, or a correction to the existing features:

$$
F(X) = H(X) - X
$$

Then the output is:

$$
Y = F(X) + X
$$

So instead of learning an entirely new representation, the convolutional layers only need to learn **what should change** about the existing representation.

---

### Intuition: editing a document

Think of it like editing a document.

**Ordinary network:**

> "Rewrite this entire document."

**Residual network:**

> "Here's the current document. Just tell me what needs to be changed."

If very little needs to change, the residual branch can learn something close to:

$$
F(X) \approx 0
$$

Then:

$$
Y = X + 0 = X
$$

The original features can therefore pass through the block essentially unchanged.

This is especially useful when stacking many residual blocks because the network always has an easy way to **preserve useful information** while the convolutional layers learn additional refinements.

---

### The key idea

$$
\boxed{\text{Residual block} = \text{learn a change to the existing features}}
$$

rather than:

$$
\boxed{\text{Residual block} = \text{learn an entirely new representation}}
$$

# 8.6.5 ResNeXt Uses Grouped Convolutions

Grouped convolution splits input channels into groups.

Each group is convolved separately, and the results are concatenated as output channels.

* With `groups=4`, a convolution with 16 input channels behaves like four smaller convolutions over 4 channels each.
* This can reduce parameters and computation while preserving multiple parallel transformation paths.
* ResNeXt uses this idea inside residual-style blocks.

    | Dense                     | Grouped                    |
    | ------------------------- | -------------------------- |
    | More channel interactions | Fewer channel interactions |
    | More parameters           | Fewer parameters           |
    | More computation          | Less computation           |
    | More expressive           | Less expressive per layer  |
    | More expensive            | Cheaper                    |


In [8]:
dense_conv = nn.Conv2d(16, 32, kernel_size=3, padding=1, groups=1, bias=False)
grouped_conv = nn.Conv2d(16, 32, kernel_size=3, padding=1, groups=4, bias=False) # Split channels into 4 groups (or 16/4=4 input_channels and 32/4=8 output_channels per group)

X = torch.randn(2, 16, 8, 8)
Y = grouped_conv(X)                                                              # output size of (8 + 2*1 - 3) + 1 = 8 for shape of (batch, out_channels, height, width) or (2, 32, 8, 8)

print("dense weight shape:", shape(dense_conv.weight))                           # (32, 16, 3, 3)
print("grouped weight shape:", shape(grouped_conv.weight))                       # (32, 4, 3, 3)
print("dense params:", count_parameters(dense_conv))                             # 32*16*3*3 = 4608
print("grouped params:", count_parameters(grouped_conv))                         # 32*4*3*3 = 1152

assert shape(Y) == (2, 32, 8, 8)
assert count_parameters(grouped_conv) < count_parameters(dense_conv)

dense weight shape: (32, 16, 3, 3)
grouped weight shape: (32, 4, 3, 3)
dense params: 4608
grouped params: 1152


# 8.6.6 Break It Deliberately: Add Tensors With Different Channels

If the main path changes channel count and the shortcut remains identity, residual addition fails.

The right fix is not to silence the error; the right fix is to add a projection shortcut or keep the shapes unchanged.

In [10]:
bad_block = ResidualBlock(8, 16, use_projection=False)

try:
    bad_block(torch.randn(2, 8, 16, 16))                                         # use_projection=True is required to transform the shortcut from 8 → 16 channels
except RuntimeError as error:
    print(type(error).__name__)
    print(str(error).splitlines()[0])
else:
    raise AssertionError("Expected residual addition to fail with channel mismatch")

RuntimeError
The size of tensor a (16) must match the size of tensor b (8) at non-singleton dimension 1


# 8.6 Checkpoint

Answer these before moving on.

Short markdown answers in the notebook are enough; the checkpoint is meant to test whether you can explain the mechanics without rereading the code.

1. What function does a residual block learn?
> A residual block learns a residual transformation `F(X)`, which is a learned modification or correction to the input features. The output is `F(X)+X`.

2. Why does residual addition require a strict shape contract?
> Residual addition requires the main branch and shortcut branch to have matching shapes because **element-wise addition operates on corresponding elements**

3. When do we need a projection shortcut?
> We need a projection shortcut when the main branch changes the shape of the input, such as changing the number of channels or spatial dimensions. The projection transforms the shortcut so it matches the main branch

4. Why can residual connections make identity mappings easier to represent?
> Because the residual branch can learn `F(X)≈0`, allowing the shortcut to pass the input through unchanged. Therefore the block can easily represent `Y=X`

5. What does grouped convolution change compared with a dense convolution?
> Grouped convolution splits input and output channels into independent groups, so each output only connects to a subset of the input channels. This reduces parameters and computation compared with dense convolution, at the cost of reduced cross-channel connectivity